# Подготовка данных для дашборда Power BI
Логика: сначала сохраняем финальный df_data с кластерами, затем строим справочники (dim) и таблицы фактов (fact). Каждый файл — отдельный CSV. В конце — проверка сходимости.


## Зачем этот ноутбук

В основной части проекта мы разобрали данные о наблюдениях за лебедями в Северной Америке: очистили их, обогатили температурой и плотностью населения, разбили по кластерам и сезонам, посчитали регрессии. Всё это осталось в исследовательском ноутбуке — с длинными таблицами, техническими проверками и промежуточными расчётами. Для дальнейшей работы с результатами это неудобно: чтобы посмотреть, где какой вид встречается и как менялось распределение, приходится перелистывать десятки ячеек.

Здесь я готовлю другой продукт. Не отчёт с цифрами, а **данные для интерактивного дашборда**, который сможет открыть любой человек — без Python, без знания статистики, без чтения наших выводов. Просто выбрал вид, сезон, регион и посмотрел на карту.

## Что хочется получить в итоге

Одностраничный отчёт в Power BI, построенный вокруг одного вопроса: **где наблюдали разных лебедей зимой и летом и как это распределение менялось со временем**. Карта с переключением режимов, четыре карточки с ключевыми цифрами, график динамики широты, сравнение видов по кластерам и компактный блок с коэффициентами регрессий как поясняющий элемент. Не центр страницы, а дополнение: сначала карта и динамика, потом уже статистика.

Дашборд должен сам объяснять, что он показывает. Чтобы человек, открывший его впервые, не запутался в терминах «наблюдения», «ячейки», «агрегированные записи» и понимал, что цифры отражают записи о встречах, а не численность птиц.

## Что нужно для этого сделать

Power BI работает с плоскими таблицами и связями между ними. Поэтому я раскладываю наш `df_data` в **звёздную модель**: одну таблицу фактов и несколько справочников.

**Таблица фактов** — `fact_observations`. Одна строка — агрегированное наблюдение вида за конкретный месяц в конкретной координатной ячейке и регионе. Плюс отдельная таблица `fact_regression` с результатами регрессий: по одной строке на каждую комбинацию «кластер × вид × сезон × фактор» с коэффициентом, доверительным интервалом и метаданными модели.

**Справочники** — время, виды, сезоны, места, кластеры. Они нужны, чтобы в дашборде можно было фильтровать и группировать данные, не дублируя текстовые поля в каждой строке факта.

Отдельно проверяю, что суммы сходятся: `ObservationCount` в сумме даёт 1 289 670, строк в факте — 70 973, уникальных координатных ячеек — 1 756. Если где-то расходится — значит, при экспорте потерялись строки или сломались ключи.

## О чём я помню при подготовке

Есть два разных показателя, и их нельзя смешивать. **Наблюдения** — сколько раз лебедя видели. **Ячейки** — в скольких координатных квадратах его отметили. Первое сильно зависит от активности людей, второе — ближе к реальному распространению. Для дашборда я использую ячейки там, где важно распространение, и наблюдения — там, где важно показать объём данных. В подсказках это указано явно.

Также помню, что регрессии в `fact_regression` **не пересчитываются** при смене фильтров в Power BI. Они построены один раз на всём периоде 1980–2023. Поэтому модельный блок реагирует только на вид, сезон и кластер — и рядом с ним всегда написано, что фильтр годов и штатов на него не влияет.

## Что должно получиться

На выходе — семь CSV-файлов, готовых к загрузке в Power BI:

- `fact_observations.csv`
- `fact_regression.csv`
- `dim_month.csv`
- `dim_species.csv`
- `dim_season.csv`
- `dim_location.csv`
- `dim_cluster.csv`

С проверенными ключами, корректными типами и контрольными суммами. Дальше — только загрузка, связи между таблицами и настройка визуалов. Никаких пересчётов в Power BI не потребуется.

План: сохранение мастер-датасета, сборка справочников, формирование таблиц фактов и финальная проверка сходимости.

## Шаг 0. Загрузка

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from IPython.display import display

In [1]:
from pathlib import Path
import pandas as pd

FILE, LEVELS = "data_seasonal_enriched.csv", 3
roots = [Path.cwd(), *Path.cwd().parents][:LEVELS + 1]

path = next((
    f for r in roots
    for f in ([r / FILE] if (r / FILE).is_file() else r.rglob(FILE))
    if len(f.relative_to(r).parts) <= LEVELS + 1
), None)

df_data = pd.read_csv(path)
print("Размер датасета:", df_data.shape)
print("Столбцы:", df_data.columns.tolist())

Размер датасета: (70973, 15)
Столбцы: ['year', 'period_type', 'decimal_latitude', 'decimal_longitude', 'species', 'state_province', 'country_code', 'time_period', 'month', 'avg_temp_c', 'land_area', 'density_per', 'observations', 'cluster', 'cluster_name']


## Шаг 1. dim_species — справочник видов

In [3]:
# ============================================================
# dim_species.csv
# ============================================================
dim_species = pd.DataFrame({
    'SpeciesID': [1, 2, 3],
    'ScientificName': ['Cygnus olor', 'Cygnus buccinator', 'Cygnus columbianus'],
    'RussianName': ['Лебедь-шипун', 'Лебедь-трубач', 'Тундровый лебедь']
})
dim_species.to_csv('dim_species.csv', index=False)
display(dim_species)

,SpeciesID,ScientificName,RussianName
0,1,Cygnus olor,Лебедь-шипун
1,2,Cygnus buccinator,Лебедь-трубач
2,3,Cygnus columbianus,Тундровый лебедь


## Шаг 2. dim_season — справочник сезонов

In [4]:
# ============================================================
# dim_season.csv
# ============================================================
dim_season = pd.DataFrame({
    'SeasonID': [1, 2],
    'SeasonName': ['Зимний период', 'Летний период'],
    'Description': [
        'Декабрь–февраль',
        'Июнь–август (приближение к сезону гнездования; факт гнездования не проверялся)'
    ]
})
dim_season.to_csv('dim_season.csv', index=False)
display(dim_season)

,SeasonID,SeasonName,Description
0,1,Зимний период,Декабрь–февраль
1,2,Летний период,Июнь–август (приближение к сезону гнездования;...


## Шаг 3. dim_cluster — справочник кластеров


In [5]:
# ============================================================
# dim_cluster.csv
# ============================================================
dim_cluster = (
    df_data[['cluster', 'cluster_name']]
    .drop_duplicates()
    .rename(columns={'cluster': 'ClusterID', 'cluster_name': 'ClusterName'})
    .sort_values('ClusterID')
    .reset_index(drop=True)
)
dim_cluster.to_csv('dim_cluster.csv', index=False)
display(dim_cluster)

,ClusterID,ClusterName
0,1,Север (Аляска и север Канады)
1,2,Юг США
2,3,Северо-восток США и восток Канады
3,4,Запад США и Канады
4,5,Центр Канады


## Шаг 4. dim_month — справочник месяцев и временных периодов

In [6]:
# ============================================================
# dim_month.csv
# ============================================================
month_names_ru = {
    1: 'Январь', 2: 'Февраль', 3: 'Март', 4: 'Апрель',
    5: 'Май', 6: 'Июнь', 7: 'Июль', 8: 'Август',
    9: 'Сентябрь', 10: 'Октябрь', 11: 'Ноябрь', 12: 'Декабрь'
}

dim_month = df_data[['year', 'month', 'time_period']].drop_duplicates().copy()
dim_month['MonthDate'] = pd.to_datetime(
    dict(year=dim_month['year'], month=dim_month['month'], day=1)
)
dim_month['MonthNumber'] = dim_month['month']
dim_month['MonthName'] = dim_month['month'].map(month_names_ru)
dim_month['TimePeriod'] = dim_month['time_period'].astype(str)

dim_month = dim_month[[
    'MonthDate', 'year', 'MonthNumber', 'MonthName', 'TimePeriod'
]].rename(columns={'year': 'Year'}).sort_values('MonthDate').reset_index(drop=True)

dim_month.to_csv('dim_month.csv', index=False)
display(dim_month.head(10))
print("Всего строк в dim_month:", len(dim_month))

,MonthDate,Year,MonthNumber,MonthName,TimePeriod
0,1980-01-01,1980,1,Январь,1_1980-1988
1,1980-02-01,1980,2,Февраль,1_1980-1988
2,1980-06-01,1980,6,Июнь,1_1980-1988
3,1980-07-01,1980,7,Июль,1_1980-1988
4,1980-08-01,1980,8,Август,1_1980-1988
5,1980-12-01,1980,12,Декабрь,1_1980-1988
6,1981-01-01,1981,1,Январь,1_1980-1988
7,1981-02-01,1981,2,Февраль,1_1980-1988
8,1981-06-01,1981,6,Июнь,1_1980-1988
9,1981-07-01,1981,7,Июль,1_1980-1988


Всего строк в dim_month: 264


## Шаг 5. dim_location — справочник мест


In [7]:
# ============================================================
# dim_location.csv
# ============================================================
dim_location = df_data[[
    'decimal_latitude', 'decimal_longitude',
    'state_province', 'country_code',
    'density_per', 'land_area'
]].drop_duplicates().copy()

# GridCellID — только по координатам
dim_location['GridCellID'] = (
    dim_location['decimal_latitude'].astype(str) + '_' +
    dim_location['decimal_longitude'].astype(str)
)

# LocationID — координаты + регион
dim_location['LocationID'] = (
    dim_location['decimal_latitude'].astype(str) + '_' +
    dim_location['decimal_longitude'].astype(str) + '_' +
    dim_location['state_province'].astype(str)
)

dim_location = dim_location[[
    'LocationID', 'GridCellID',
    'decimal_latitude', 'decimal_longitude',
    'state_province', 'country_code',
    'density_per', 'land_area'
]].rename(columns={
    'decimal_latitude': 'Latitude',
    'decimal_longitude': 'Longitude',
    'state_province': 'StateProvince',
    'country_code': 'Country',
    'density_per': 'DensityPer',
    'land_area': 'LandArea'
}).reset_index(drop=True)

dim_location.to_csv('dim_location.csv', index=False)
display(dim_location.head())
print("Уникальных мест (LocationID):", dim_location['LocationID'].nunique())
print("Уникальных координатных ячеек (GridCellID):", dim_location['GridCellID'].nunique())

,LocationID,GridCellID,Latitude,Longitude,StateProvince,Country,DensityPer,LandArea
0,33_-82_Georgia,33_-82,33,-82,Georgia,US,74.0,148959.0
1,34_-118_California,34_-118,34,-118,California,US,97.0,403466.0
2,34_-79_South Carolina,34_-79,34,-79,South Carolina,US,69.0,77857.0
3,35_-76_North Carolina,35_-76,35,-76,North Carolina,US,86.0,125920.0
4,36_-84_Tennessee,36_-84,36,-84,Tennessee,US,67.0,106798.0


Уникальных мест (LocationID): 2101
Уникальных координатных ячеек (GridCellID): 1756


## Шаг 6. fact_observations — основная таблица фактов


In [8]:
# ============================================================
# fact_observations.csv
# ============================================================
fact = df_data.copy()

# MonthDate
fact['MonthDate'] = pd.to_datetime(
    dict(year=fact['year'], month=fact['month'], day=1)
)

# SpeciesID
species_map = {row['ScientificName']: row['SpeciesID']
               for _, row in dim_species.iterrows()}
fact['SpeciesID'] = fact['species'].map(species_map)

# SeasonID = period_type
fact['SeasonID'] = fact['period_type'].astype(int)

# LocationID
fact['LocationID'] = (
    fact['decimal_latitude'].astype(str) + '_' +
    fact['decimal_longitude'].astype(str) + '_' +
    fact['state_province'].astype(str)
)

# ClusterID
fact['ClusterID'] = fact['cluster'].astype(int)

# ObservationCount
fact['ObservationCount'] = fact['observations'].astype(int)

# AvgTempC
fact['AvgTempC'] = fact['avg_temp_c'].astype(float)

# Финальный набор столбцов
fact_observations = fact[[
    'MonthDate', 'SpeciesID', 'SeasonID', 'LocationID',
    'ClusterID', 'AvgTempC', 'ObservationCount'
]].reset_index(drop=True)

fact_observations.to_csv('fact_observations.csv', index=False)
display(fact_observations.head())
print("Строк в факте:", len(fact_observations))

,MonthDate,SpeciesID,SeasonID,LocationID,ClusterID,AvgTempC,ObservationCount
0,1980-12-01,3,1,33_-82_Georgia,2,7.611111,2
1,1980-02-01,1,1,34_-118_California,2,9.333333,1
2,1980-01-01,3,1,34_-79_South Carolina,3,7.111111,1
3,1980-02-01,3,1,34_-79_South Carolina,3,5.388889,1
4,1980-12-01,3,1,34_-79_South Carolina,3,6.888889,1


Строк в факте: 70973


## Шаг 7. fact_regression — таблица коэффициентов регрессии


In [9]:
# ============================================================
# fact_regression.csv
# ============================================================
import statsmodels.api as sm

species_map_inv = {v: k for k, v in species_map.items()}  # SpeciesID -> name
cluster_map_inv = dict(zip(dim_cluster['ClusterID'], dim_cluster['ClusterName']))
season_map = {1: 'Зимний период', 2: 'Летний период'}

def fit_regression(subset):
    predictors = ['year', 'avg_temp_c', 'density_per']

    required = [
        'decimal_latitude',
        'decimal_longitude',
        *predictors
    ]

    work = subset.dropna(subset=required).copy()

    # Сохраняем прежний технический порог.
    if len(work) < 50:
        return None

    X = sm.add_constant(
        work[predictors].astype('float64'),
        has_constant='add'
    )

    y = work['decimal_latitude'].astype('float64')

    # Проверка возможности оценить модель.
    if y.nunique() < 2:
        return None

    if len(work) <= X.shape[1]:
        return None

    if np.linalg.matrix_rank(X.to_numpy()) < X.shape[1]:
        return None

    # Координатная ячейка — только широта и долгота.
    # Административный регион в этот ключ не входит.
    cell_index = pd.MultiIndex.from_frame(
        work[['decimal_latitude', 'decimal_longitude']]
    )

    cell_codes, unique_cells = pd.factorize(cell_index)

    if len(unique_cells) < 2:
        return None

    return sm.OLS(y, X).fit(
        cov_type='cluster',
        cov_kwds={
            'groups': cell_codes,
            'use_correction': True,
            'df_correction': True
        },
        use_t=True
    )
rows = []
factors = [
    ('year', 'Год'),
    ('avg_temp_c', 'Температура'),
    ('density_per', 'Плотность')
]

for cid, cname in cluster_map_inv.items():
    for sid, sname in species_map_inv.items():
        for season_id, season_name in season_map.items():
            subset = df_data[
                (df_data['cluster'] == cid) &
                (df_data['species'] == sname) &
                (df_data['period_type'] == season_id)
            ]
            model = fit_regression(subset)

            if model is None:
                # Модель не строилась
                for factor_code, factor_name in factors:
                    rows.append({
                        'ClusterID': cid,
                        'ClusterName': cname,
                        'SpeciesID': sid,
                        'SeasonID': season_id,
                        'SeasonName': season_name,
                        'Factor': factor_name,
                        'Coefficient': None,
                        'CILow': None,
                        'CIHigh': None,
                        'PValue': None,
                        'R2': None,
                        'NRows': len(subset),
                        'ModelStartYear': None,
                        'ModelEndYear': None,
                        'ModelType': 'Недостаточно данных'
                    })
                continue

            r2 = model.rsquared
            ci = model.conf_int()
            n = int(model.nobs)
            y_min = int(subset['year'].min())
            y_max = int(subset['year'].max())

            for factor_code, factor_name in factors:
                rows.append({
                    'ClusterID': cid,
                    'ClusterName': cname,
                    'SpeciesID': sid,
                    'SeasonID': season_id,
                    'SeasonName': season_name,
                    'Factor': factor_name,
                    'Coefficient': round(model.params[factor_code], 6),
                    'CILow': round(ci.loc[factor_code, 0], 6),
                    'CIHigh': round(ci.loc[factor_code, 1], 6),
                    'PValue': model.pvalues[factor_code],
                    'R2': round(r2, 4),
                    'NRows': n,
                    'ModelStartYear': y_min,
                    'ModelEndYear': y_max,
                    'ModelType': 'Множественная OLS'
                })

fact_regression = pd.DataFrame(rows)
fact_regression.to_csv('fact_regression.csv', index=False)

display(fact_regression.head(20))
print("Всего строк в fact_regression:", len(fact_regression))

,ClusterID,ClusterName,SpeciesID,SeasonID,SeasonName,Factor,Coefficient,CILow,CIHigh,PValue,R2,NRows,ModelStartYear,ModelEndYear,ModelType
0,1,Север (Аляска и север Канады),1,1,Зимний период,Год,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,Недостаточно данных
1,1,Север (Аляска и север Канады),1,1,Зимний период,Температура,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,Недостаточно данных
2,1,Север (Аляска и север Канады),1,1,Зимний период,Плотность,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,Недостаточно данных
3,1,Север (Аляска и север Канады),1,2,Летний период,Год,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,Недостаточно данных
4,1,Север (Аляска и север Канады),1,2,Летний период,Температура,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,Недостаточно данных
5,1,Север (Аляска и север Канады),1,2,Летний период,Плотность,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,Недостаточно данных
6,1,Север (Аляска и север Канады),2,1,Зимний период,Год,0.039170,0.007802,0.070538,1.542690e-02,0.6330,616,1981.0,2023.0,Множественная OLS
7,1,Север (Аляска и север Канады),2,1,Зимний период,Температура,-0.001841,-0.040851,0.037169,9.248611e-01,0.6330,616,1981.0,2023.0,Множественная OLS
8,1,Север (Аляска и север Канады),2,1,Зимний период,Плотность,-0.830585,-1.074371,-0.586799,1.055805e-08,0.6330,616,1981.0,2023.0,Множественная OLS
9,1,Север (Аляска и север Канады),2,2,Летний период,Год,0.002017,-0.012176,0.016210,7.797969e-01,0.1867,2511,1980.0,2023.0,Множественная OLS


Всего строк в fact_regression: 90


## Шаг 8. Проверка сходимости


In [10]:
# ============================================================
# Проверка контрольных сумм
# ============================================================

print("=" * 60)
print("ПРОВЕРКА СООТВЕТСТВИЯ")
print("=" * 60)

# Строки факта наблюдений
print(f"Строк в fact_observations: {len(fact_observations)} (ожидается 70 973)")

# Сумма ObservationCount
total_obs = fact_observations['ObservationCount'].sum()
print(f"Сумма ObservationCount: {total_obs:,} (ожидается 1 289 670)")

# Уникальные координатные ячейки
n_grid = dim_location['GridCellID'].nunique()
print(f"Уникальных координатных ячеек: {n_grid} (ожидается 1 756)")

# Уникальные места
print(f"Уникальных LocationID: {dim_location['LocationID'].nunique()}")

# Проверка ключей справочников
print()
print("Уникальность ключей:")
print(f"  SpeciesID уникальны: {dim_species['SpeciesID'].is_unique}")
print(f"  SeasonID уникальны: {dim_season['SeasonID'].is_unique}")
print(f"  ClusterID уникальны: {dim_cluster['ClusterID'].is_unique}")
print(f"  LocationID уникальны: {dim_location['LocationID'].is_unique}")
print(f"  MonthDate уникальны: {dim_month['MonthDate'].is_unique}")

# Проверка отсутствия пропусков в ключах
print()
print("Пропуски в ключах fact_observations:")
print(fact_observations[['MonthDate', 'SpeciesID', 'SeasonID',
                          'LocationID', 'ClusterID']].isna().sum())

ПРОВЕРКА СООТВЕТСТВИЯ
Строк в fact_observations: 70973 (ожидается 70 973)
Сумма ObservationCount: 1,289,670 (ожидается 1 289 670)
Уникальных координатных ячеек: 1756 (ожидается 1 756)
Уникальных LocationID: 2101

Уникальность ключей:
  SpeciesID уникальны: True
  SeasonID уникальны: True
  ClusterID уникальны: True
  LocationID уникальны: True
  MonthDate уникальны: True

Пропуски в ключах fact_observations:
MonthDate     0
SpeciesID     0
SeasonID      0
LocationID    0
ClusterID     0
dtype: int64
